# DAC 2026 V5 — Repaired Supervised Boundary Stack

**Kaggle target:** `NormalizedRecall@5%` only.  
**Runtime target:** Kaggle Notebook, **T4 x2**.  
**Competition-integrity boundary:** the pipeline uses only the official `train.csv`, `test.csv`, and `sample_submission.csv` discovered under `/kaggle/input`.

V5 is the evidence-led continuation of V1–V4:

- retain V2's order-safe grouped validation, semantic categorical handling, and fold-bagging;
- restore V3's **cross-fitted supervised stacker** and **supervised high-risk refiner**, the only architecture change that produced a material boundary gain;
- repair V3's validation/deployment mismatch by bagging the complete meta/refiner pipeline;
- keep V4's multi-scale empirical-risk evidence, but use it as **features**, never as a deterministic direct reranker;
- evaluate the risky meta/refiner stage on **two independent group-aware partitions**;
- select exclusively on held-out `NormalizedRecall@5%`, with stability diagnostics.

No row position, external data, old predictions, hidden-label helpers, public-leaderboard outcome, pretrained external model weights, or `claim_id` predictive feature is used.

## 1. Configuration

The expensive base layer is trained once with five exact-signature grouped folds. Two independent meta/refiner CV seeds then stress-test the stage most vulnerable to selection overfit without duplicating the entire boosted-tree workload.

Candidate refiner pools are intentionally compact: 8%, 10%, and 12%. Refiner blending may be aggressive when held-out evidence supports it, including weight 1.0. The notebook does not assume that conservative reranking is automatically safer after V4 showed otherwise.

In [1]:
from __future__ import annotations

import gc
import json
import math
import os
import random
import hashlib
import warnings
from dataclasses import dataclass
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, brier_score_loss
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier

warnings.filterwarnings('ignore')

SEED = 2026
BASE_FOLDS = 5
META_SEEDS = [2026, 9117]
META_FOLDS = 5
INNER_META_FOLDS = 4
CAPACITY = 0.05
CANDIDATE_RATES = [0.08, 0.10, 0.12]
REFINER_WEIGHTS = [0.65, 0.85, 1.00]
EB_ALPHAS = [1.0, 3.0, 10.0, 30.0]
SUBMISSION_NR_GATE = 0.9815
SUBMISSION_WIN_GATE = 7

OUT_DIR = Path('/kaggle/working/v5_artifacts')
OUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)

print('V5 configuration loaded.')
print('Base folds:', BASE_FOLDS, '| meta seeds:', META_SEEDS, '| candidate rates:', CANDIDATE_RATES)

V5 configuration loaded.
Base folds: 5 | meta seeds: [2026, 9117] | candidate rates: [0.08, 0.1, 0.12]


## 2. Official-data discovery and compliance preflight

The discovery logic only accepts a directory containing all three official competition files with the expected schema. Paths containing `ground_truth`, previous results, submissions, or artifact folders are ignored.

In [2]:
def discover_official_data(root: str = '/kaggle/input'):
    root = Path(root)
    forbidden_tokens = {
        'ground_truth', 'result', 'artifact', 'submission_v', 'prediction',
        'notebook', 'handoff', 'answer sheet', 'infomagang'
    }
    candidates = []
    for train_path in root.rglob('train.csv'):
        parent = train_path.parent
        text = str(parent).lower()
        if any(tok in text for tok in forbidden_tokens):
            continue
        test_path = parent / 'test.csv'
        sample_path = parent / 'sample_submission.csv'
        if not test_path.exists() or not sample_path.exists():
            continue
        try:
            tr_head = pd.read_csv(train_path, nrows=5)
            te_head = pd.read_csv(test_path, nrows=5)
            sa_head = pd.read_csv(sample_path, nrows=5)
        except Exception:
            continue
        if 'label' not in tr_head.columns or 'label' in te_head.columns:
            continue
        if 'claim_id' not in tr_head.columns or 'claim_id' not in te_head.columns:
            continue
        if list(sa_head.columns) != ['claim_id', 'fraud_probability']:
            continue
        candidates.append((train_path, test_path, sample_path))
    if not candidates:
        raise FileNotFoundError('No valid official train/test/sample_submission trio found under /kaggle/input.')
    candidates = sorted(candidates, key=lambda x: str(x[0]))
    if len(candidates) > 1:
        print('Multiple valid trios detected. Using:', candidates[0][0].parent)
    return candidates[0]

TRAIN_PATH, TEST_PATH, SAMPLE_PATH = discover_official_data()
print('train :', TRAIN_PATH)
print('test  :', TEST_PATH)
print('sample:', SAMPLE_PATH)

# Preserve categorical identifiers with leading zeroes from the CSV boundary.
dtype_map = {
    'claim_id': 'string',
    'kdkc': 'string',
    'dati2': 'string',
    'typeppk': 'string',
    'jkpst': 'string',
    'jnspelsep': 'string',
    'cmg': 'string',
    'diagprimer': 'string',
}
train_raw = pd.read_csv(TRAIN_PATH, dtype=dtype_map)
test_raw = pd.read_csv(TEST_PATH, dtype=dtype_map)
sample = pd.read_csv(SAMPLE_PATH, dtype={'claim_id': 'string'})

assert len(train_raw) > 0 and len(test_raw) > 0
assert 'label' in train_raw.columns and 'label' not in test_raw.columns
assert set(train_raw['label'].dropna().unique()).issubset({0, 1})
assert train_raw['claim_id'].is_unique and test_raw['claim_id'].is_unique
assert train_raw['claim_id'].isna().sum() == 0 and test_raw['claim_id'].isna().sum() == 0
assert list(sample.columns) == ['claim_id', 'fraud_probability']
assert len(sample) == len(test_raw)

# Align test to the official sample order without using claim_id as a feature.
if not sample['claim_id'].reset_index(drop=True).equals(test_raw['claim_id'].reset_index(drop=True)):
    test_raw = sample[['claim_id']].merge(test_raw, on='claim_id', how='left', validate='one_to_one')
    assert len(test_raw) == len(sample) and test_raw.notna().all(axis=None)

print('Train shape:', train_raw.shape, '| Test shape:', test_raw.shape)
print('Fraud prevalence:', float(train_raw['label'].mean()))

train : /kaggle/input/competitions/dac-its-2026/train.csv
test  : /kaggle/input/competitions/dac-its-2026/test.csv
sample: /kaggle/input/competitions/dac-its-2026/sample_submission.csv
Train shape: (160174, 53) | Test shape: (40043, 52)
Fraud prevalence: 0.5007242124189943


## 3. Exact competition metric and label-neutral tie handling

`claim_id` is never a predictive feature. A deterministic cryptographic hash is used only as a secondary key when exact score ties cross the audit boundary, preventing source-row order from deciding the metric.

In [3]:
def make_tie_key(ids: pd.Series) -> np.ndarray:
    out = np.empty(len(ids), dtype=np.uint64)
    for i, x in enumerate(ids.astype(str).tolist()):
        digest = hashlib.blake2b(('DAC2026_V5_TIE|' + x).encode(), digest_size=8).digest()
        out[i] = int.from_bytes(digest, byteorder='little', signed=False)
    return out

TRAIN_TIE = make_tie_key(train_raw['claim_id'])
TEST_TIE = make_tie_key(test_raw['claim_id'])


def topk_indices(scores: np.ndarray, tie_key: np.ndarray, capacity: float = CAPACITY) -> np.ndarray:
    scores = np.asarray(scores, dtype=float)
    if not np.isfinite(scores).all():
        raise ValueError('Non-finite scores encountered.')
    k = int(math.floor(capacity * len(scores)))
    if k <= 0:
        return np.array([], dtype=int)
    order = np.lexsort((tie_key, -scores))
    return order[:k]


def nr_at_capacity(y_true, scores, tie_key, capacity: float = CAPACITY):
    y_true = np.asarray(y_true, dtype=int)
    idx = topk_indices(scores, tie_key, capacity)
    k = len(idx)
    p = int(y_true.sum())
    if k == 0 or p == 0:
        return {'nr': 0.0, 'tp': 0, 'k': k, 'p': p, 'precision': 0.0}
    tp = int(y_true[idx].sum())
    recall = tp / p
    recall_max = min(1.0, k / p)
    nr = recall / recall_max if recall_max > 0 else 0.0
    return {'nr': float(nr), 'tp': tp, 'k': k, 'p': p, 'precision': float(tp / k)}


def pct_rank(scores: np.ndarray) -> np.ndarray:
    return pd.Series(np.asarray(scores, dtype=float)).rank(method='average', pct=True).to_numpy(dtype=np.float64)

# Metric sanity checks.
y_demo = np.array([1, 0, 1, 0, 1, 0, 1, 0, 1, 0] * 20)
s_demo = np.linspace(1, 0, len(y_demo))
t_demo = np.arange(len(y_demo), dtype=np.uint64)
assert 0 <= nr_at_capacity(y_demo, s_demo, t_demo)['nr'] <= 1
print('Metric sanity checks passed.')

Metric sanity checks passed.


## 4. Order-safe feature engineering

Administrative codes remain categorical. No numeric-prefix hierarchy is assumed for anonymized identifiers. V5 retains the burden, concentration, density, and clinically interpretable interaction families that repeatedly mattered in V1–V3.

In [4]:
def entropy_from_counts(arr: np.ndarray) -> np.ndarray:
    arr = np.asarray(arr, dtype=np.float64)
    row_sum = arr.sum(axis=1, keepdims=True)
    p = np.divide(arr, row_sum, out=np.zeros_like(arr), where=row_sum > 0)
    logp = np.zeros_like(p)
    mask = p > 0
    logp[mask] = np.log(p[mask])
    return -(p * logp).sum(axis=1)


def engineer_features(df: pd.DataFrame) -> pd.DataFrame:
    x = df.drop(columns=['label'], errors='ignore').copy()
    x = x.drop(columns=['claim_id'], errors='ignore')

    for c in ['kdkc', 'dati2', 'typeppk', 'jkpst', 'jnspelsep', 'cmg', 'diagprimer']:
        if c in x.columns:
            x[c] = x[c].astype('string').fillna('__MISSING__').str.strip().astype(str)

    # Preserve severity numerically, but expose a categorical view for interactions.
    x['severity_cat'] = x['severitylevel'].astype(str)
    x['age_band'] = pd.cut(
        x['umur'], bins=[-np.inf, 17, 39, 59, np.inf],
        labels=['0_17', '18_39', '40_59', '60_plus']
    ).astype(str)
    x['is_pediatric'] = (x['umur'] <= 17).astype(np.int8)
    x['is_older_adult'] = (x['umur'] >= 60).astype(np.int8)

    los = x['los'].astype(float).clip(lower=0)
    x['los_zero'] = (los == 0).astype(np.int8)
    x['log1p_los'] = np.log1p(los)
    x['age_x_loglos'] = x['umur'].astype(float) * x['log1p_los']

    dx_cols = [c for c in x.columns if c.startswith('dx2_')]
    proc_cols = [c for c in x.columns if c.startswith('proc')]
    dx_arr = x[dx_cols].to_numpy(dtype=np.float64) if dx_cols else np.zeros((len(x), 1))
    pr_arr = x[proc_cols].to_numpy(dtype=np.float64) if proc_cols else np.zeros((len(x), 1))

    x['diagnosis_total'] = dx_arr.sum(axis=1)
    x['diagnosis_nonzero_groups'] = (dx_arr > 0).sum(axis=1)
    x['diagnosis_max'] = dx_arr.max(axis=1)
    x['diagnosis_entropy'] = entropy_from_counts(dx_arr)
    x['diagnosis_max_share'] = np.divide(
        x['diagnosis_max'], x['diagnosis_total'], out=np.zeros(len(x)), where=x['diagnosis_total'].to_numpy() > 0
    )

    x['procedure_total'] = pr_arr.sum(axis=1)
    x['procedure_nonzero_groups'] = (pr_arr > 0).sum(axis=1)
    x['procedure_max'] = pr_arr.max(axis=1)
    x['procedure_entropy'] = entropy_from_counts(pr_arr)
    x['procedure_max_share'] = np.divide(
        x['procedure_max'], x['procedure_total'], out=np.zeros(len(x)), where=x['procedure_total'].to_numpy() > 0
    )

    x['coded_burden'] = x['diagnosis_total'] + x['procedure_total']
    x['log1p_coded_burden'] = np.log1p(x['coded_burden'])
    x['total_nonzero_groups'] = x['diagnosis_nonzero_groups'] + x['procedure_nonzero_groups']
    denom_days = np.maximum(los.to_numpy(), 1.0)
    x['burden_per_day'] = x['coded_burden'].to_numpy() / denom_days
    x['diagnosis_per_day'] = x['diagnosis_total'].to_numpy() / denom_days
    x['procedure_per_day'] = x['procedure_total'].to_numpy() / denom_days
    x['procedure_to_diagnosis'] = (x['procedure_total'] + 0.5) / (x['diagnosis_total'] + 0.5)

    def cross(a, b):
        return x[a].astype(str).str.cat(x[b].astype(str), sep='__')

    x['geo_facility'] = cross('dati2', 'typeppk')
    x['kdkc_dati2'] = cross('kdkc', 'dati2')
    x['cmg_severity'] = cross('cmg', 'severity_cat')
    x['diagnosis_service'] = cross('diagprimer', 'jnspelsep')
    x['diagnosis_cmg'] = cross('diagprimer', 'cmg')
    x['diagnosis_facility'] = cross('diagprimer', 'typeppk')
    x['cmg_facility'] = cross('cmg', 'typeppk')

    return x

X = engineer_features(train_raw)
X_test = engineer_features(test_raw)
y = train_raw['label'].to_numpy(dtype=np.int8)

# Remove features constant in official training data only.
constant_cols = [c for c in X.columns if X[c].nunique(dropna=False) <= 1]
X = X.drop(columns=constant_cols)
X_test = X_test.drop(columns=constant_cols, errors='ignore')
assert list(X.columns) == list(X_test.columns)

cat_cols = [c for c in X.columns if str(X[c].dtype) in ('object', 'string')]
for c in cat_cols:
    X[c] = X[c].astype(str)
    X_test[c] = X_test[c].astype(str)
num_cols = [c for c in X.columns if c not in cat_cols]

print('Engineered features:', X.shape[1], '| categorical:', len(cat_cols), '| numeric:', len(num_cols))
print('Dropped constants:', constant_cols)

Engineered features: 79 | categorical: 16 | numeric: 63
Dropped constants: ['dx2_koo_k93', 'dx2_u00_u99', 'procv00_v89']


## 5. Exact-signature grouped folds

Exact repeated feature signatures are kept within one fold. Training rows are additionally permuted with a fixed seed before every fit so the known storage-order defect cannot influence the learner through fit order.

In [5]:
# Signature uses model features only, never label, row index, or claim_id.
signature = pd.util.hash_pandas_object(X, index=False).astype(str).to_numpy()
base_cv = StratifiedGroupKFold(n_splits=BASE_FOLDS, shuffle=True, random_state=SEED)
base_splits = list(base_cv.split(X, y, groups=signature))

fold_rows = []
for f, (tr, va) in enumerate(base_splits, 1):
    overlap = len(set(signature[tr]).intersection(set(signature[va])))
    assert overlap == 0
    fold_rows.append({
        'fold': f, 'train_n': len(tr), 'valid_n': len(va),
        'valid_fraud_rate': float(y[va].mean()), 'signature_overlap': overlap
    })
base_fold_table = pd.DataFrame(fold_rows)
display(base_fold_table)
base_fold_table.to_csv(OUT_DIR / 'base_folds.csv', index=False)

,fold,train_n,valid_n,valid_fraud_rate,signature_overlap
0,1,128195,31979,0.496545,0
1,2,128037,32137,0.503065,0
2,3,128264,31910,0.499123,0
3,4,128132,32042,0.502934,0
4,5,128068,32106,0.501931,0


## 6. Fold-safe multi-scale empirical-risk evidence

These are **supporting predictive features**, not deterministic rules. Every OOF value is computed from that row's outer-training fold only. Test statistics are averaged across the same five training-fold mappings.

The hierarchy set is deliberately small and based on repeatedly observed geography/facility/case-mix structure, without assuming numeric meaning in anonymized codes.

In [6]:
EB_SPECS = [
    ('dati2_typeppk', ['dati2', 'typeppk']),
    ('dati2_typeppk_cmg', ['dati2', 'typeppk', 'cmg']),
    ('dati2_typeppk_cmg_diag', ['dati2', 'typeppk', 'cmg', 'diagprimer']),
    ('dati2_typeppk_cmg_sev', ['dati2', 'typeppk', 'cmg', 'severity_cat']),
    ('dati2_typeppk_service_cmg', ['dati2', 'typeppk', 'jnspelsep', 'cmg']),
    ('kdkc_typeppk_cmg_diag', ['kdkc', 'typeppk', 'cmg', 'diagprimer']),
    ('typeppk_cmg_diag_sev', ['typeppk', 'cmg', 'diagprimer', 'severity_cat']),
]


def make_key(df: pd.DataFrame, cols: list[str]) -> pd.Series:
    s = df[cols[0]].astype(str)
    for c in cols[1:]:
        s = s.str.cat(df[c].astype(str), sep='\x1f')
    return s


def wilson_lower(pos, n, z=1.96):
    pos = np.asarray(pos, dtype=float)
    n = np.asarray(n, dtype=float)
    phat = np.divide(pos, n, out=np.zeros_like(pos), where=n > 0)
    denom = 1.0 + z*z/np.maximum(n, 1.0)
    center = phat + z*z/(2*np.maximum(n, 1.0))
    rad = z * np.sqrt(
        np.divide(phat*(1-phat), np.maximum(n, 1.0), out=np.zeros_like(phat), where=n > 0)
        + z*z/(4*np.maximum(n, 1.0)**2)
    )
    lower = (center - rad) / denom
    lower[n == 0] = 0.0
    return np.clip(lower, 0, 1)


def eb_transform(train_df, train_y, target_df, spec_name, cols, alphas):
    prior = float(np.mean(train_y))
    key_tr = make_key(train_df, cols)
    key_tg = make_key(target_df, cols)
    agg = pd.DataFrame({'key': key_tr, 'y': train_y}).groupby('key', sort=False)['y'].agg(['sum', 'count'])
    pos_map = agg['sum'].to_dict()
    cnt_map = agg['count'].to_dict()
    pos = key_tg.map(pos_map).fillna(0).to_numpy(dtype=float)
    cnt = key_tg.map(cnt_map).fillna(0).to_numpy(dtype=float)
    empirical = np.divide(pos, cnt, out=np.full(len(target_df), prior), where=cnt > 0)
    entropy = -(empirical*np.log(np.clip(empirical, 1e-9, 1)) + (1-empirical)*np.log(np.clip(1-empirical, 1e-9, 1)))
    out = {}
    for a in alphas:
        out[f'eb__{spec_name}__risk_a{int(a)}'] = (pos + a*prior) / (cnt + a)
    out[f'eb__{spec_name}__empirical'] = empirical
    out[f'eb__{spec_name}__log_support'] = np.log1p(cnt)
    out[f'eb__{spec_name}__purity'] = np.abs(empirical - 0.5) * 2.0
    out[f'eb__{spec_name}__entropy'] = entropy
    out[f'eb__{spec_name}__wilson_lower'] = wilson_lower(pos, cnt)
    return pd.DataFrame(out, index=target_df.index)

# Determine stable column order once.
_eb_probe = []
for spec_name, _ in EB_SPECS:
    for a in EB_ALPHAS:
        _eb_probe.append(f'eb__{spec_name}__risk_a{int(a)}')
    _eb_probe.extend([
        f'eb__{spec_name}__empirical', f'eb__{spec_name}__log_support',
        f'eb__{spec_name}__purity', f'eb__{spec_name}__entropy', f'eb__{spec_name}__wilson_lower'
    ])
EB_COLUMNS = _eb_probe

eb_oof = pd.DataFrame(np.zeros((len(X), len(EB_COLUMNS)), dtype=np.float32), columns=EB_COLUMNS, index=X.index)
eb_test_sum = np.zeros((len(X_test), len(EB_COLUMNS)), dtype=np.float64)

for fold, (tr_idx, va_idx) in enumerate(base_splits, 1):
    tr_df, va_df = X.iloc[tr_idx], X.iloc[va_idx]
    fold_val_parts, fold_test_parts = [], []
    for spec_name, cols in EB_SPECS:
        fold_val_parts.append(eb_transform(tr_df, y[tr_idx], va_df, spec_name, cols, EB_ALPHAS))
        fold_test_parts.append(eb_transform(tr_df, y[tr_idx], X_test, spec_name, cols, EB_ALPHAS))
    val_eb = pd.concat(fold_val_parts, axis=1)[EB_COLUMNS]
    test_eb = pd.concat(fold_test_parts, axis=1)[EB_COLUMNS]
    eb_oof.iloc[va_idx] = val_eb.to_numpy(dtype=np.float32)
    eb_test_sum += test_eb.to_numpy(dtype=np.float64)
    print(f'EB fold {fold}/{BASE_FOLDS} complete.')

eb_test = pd.DataFrame((eb_test_sum / BASE_FOLDS).astype(np.float32), columns=EB_COLUMNS, index=X_test.index)
assert np.isfinite(eb_oof.to_numpy()).all() and np.isfinite(eb_test.to_numpy()).all()
print('Empirical-risk features:', len(EB_COLUMNS))

EB fold 1/5 complete.
EB fold 2/5 complete.
EB fold 3/5 complete.
EB fold 4/5 complete.
EB fold 5/5 complete.
Empirical-risk features: 63


## 7. Base model layer

V5 restores the diverse V3 model views rather than V4's fixed two-CatBoost consensus. CatBoost remains the anchor. A second CatBoost seed is retained primarily as disagreement evidence. XGBoost, LightGBM, and a linear logistic view provide different inductive biases even when their standalone scores are lower.

In [7]:
def shuffled_indices(idx, seed):
    idx = np.asarray(idx).copy()
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)
    return idx


def make_sparse_matrices(X_tr, X_va, X_te, cat_cols, num_cols):
    enc = OneHotEncoder(handle_unknown='ignore', min_frequency=2, dtype=np.float32, sparse_output=True)
    tr_cat = enc.fit_transform(X_tr[cat_cols])
    va_cat = enc.transform(X_va[cat_cols])
    te_cat = enc.transform(X_te[cat_cols])
    tr_num = sp.csr_matrix(X_tr[num_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=np.float32))
    va_num = sp.csr_matrix(X_va[num_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=np.float32))
    te_num = sp.csr_matrix(X_te[num_cols].replace([np.inf, -np.inf], 0).fillna(0).to_numpy(dtype=np.float32))
    return (
        sp.hstack([tr_cat, tr_num], format='csr'),
        sp.hstack([va_cat, va_num], format='csr'),
        sp.hstack([te_cat, te_num], format='csr'),
    )

BASE_MODELS = ['cat_2127', 'cat_2026', 'xgb', 'lgb', 'logit']
base_oof = {m: np.zeros(len(X), dtype=np.float64) for m in BASE_MODELS}
base_test_folds = {m: [] for m in BASE_MODELS}
base_metric_rows = []

cat_common = dict(
    iterations=2860,
    learning_rate=0.03,
    depth=8,
    l2_leaf_reg=8.0,
    random_strength=0.35,
    bagging_temperature=0.5,
    max_ctr_complexity=2,
    border_count=128,
    loss_function='Logloss',
    task_type='GPU',
    devices='0:1',
    verbose=False,
    allow_writing_files=False,
)

for fold, (tr_idx_raw, va_idx) in enumerate(base_splits, 1):
    tr_idx = shuffled_indices(tr_idx_raw, SEED + 1000*fold)
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y[tr_idx], y[va_idx]

    # CatBoost anchor, seed 2127.
    cat_a = CatBoostClassifier(random_seed=2127, **cat_common)
    cat_a.fit(X_tr, y_tr, cat_features=cat_cols)
    p_va = cat_a.predict_proba(X_va)[:, 1]
    p_te = cat_a.predict_proba(X_test)[:, 1]
    base_oof['cat_2127'][va_idx] = p_va
    base_test_folds['cat_2127'].append(p_te)

    # Same architecture, independent seed. Kept as diversity evidence rather than fixed authority.
    cat_b = CatBoostClassifier(random_seed=2026, **cat_common)
    cat_b.fit(X_tr, y_tr, cat_features=cat_cols)
    p_va_b = cat_b.predict_proba(X_va)[:, 1]
    p_te_b = cat_b.predict_proba(X_test)[:, 1]
    base_oof['cat_2026'][va_idx] = p_va_b
    base_test_folds['cat_2026'].append(p_te_b)

    del cat_a, cat_b
    gc.collect()

    # One fold-fitted sparse representation shared by complementary models.
    Xtr_sp, Xva_sp, Xte_sp = make_sparse_matrices(X_tr, X_va, X_test, cat_cols, num_cols)

    xgb = XGBClassifier(
        n_estimators=2200, learning_rate=0.035, max_depth=7, min_child_weight=6,
        subsample=0.85, colsample_bytree=0.82, reg_alpha=0.08, reg_lambda=7.0,
        objective='binary:logistic', eval_metric='logloss', tree_method='hist',
        device='cuda', random_state=SEED + fold, n_jobs=4
    )
    xgb.fit(Xtr_sp, y_tr, verbose=False)
    base_oof['xgb'][va_idx] = xgb.predict_proba(Xva_sp)[:, 1]
    base_test_folds['xgb'].append(xgb.predict_proba(Xte_sp)[:, 1])
    del xgb
    gc.collect()

    lgb = LGBMClassifier(
        n_estimators=2200, learning_rate=0.025, num_leaves=48, max_depth=-1,
        min_child_samples=60, subsample=0.85, colsample_bytree=0.82,
        reg_alpha=0.10, reg_lambda=7.0, objective='binary',
        random_state=SEED + 100 + fold, n_jobs=-1, verbosity=-1
    )
    lgb.fit(Xtr_sp, y_tr)
    base_oof['lgb'][va_idx] = lgb.predict_proba(Xva_sp)[:, 1]
    base_test_folds['lgb'].append(lgb.predict_proba(Xte_sp)[:, 1])
    del lgb
    gc.collect()

    logit = LogisticRegression(C=0.35, solver='liblinear', max_iter=220, random_state=SEED + fold)
    logit.fit(Xtr_sp, y_tr)
    base_oof['logit'][va_idx] = logit.predict_proba(Xva_sp)[:, 1]
    base_test_folds['logit'].append(logit.predict_proba(Xte_sp)[:, 1])
    del logit, Xtr_sp, Xva_sp, Xte_sp
    gc.collect()

    for m in BASE_MODELS:
        met = nr_at_capacity(y_va, base_oof[m][va_idx], TRAIN_TIE[va_idx])
        base_metric_rows.append({'fold': fold, 'model': m, **met})
    print(f'Base fold {fold}/{BASE_FOLDS} complete.')

base_metrics = pd.DataFrame(base_metric_rows)
base_metrics.to_csv(OUT_DIR / 'base_fold_metrics.csv', index=False)
display(base_metrics.pivot(index='fold', columns='model', values='nr'))

base_test = {m: np.mean(np.vstack(base_test_folds[m]), axis=0) for m in BASE_MODELS}
for m in BASE_MODELS:
    overall = nr_at_capacity(y, base_oof[m], TRAIN_TIE)
    print(m, overall)

Base fold 1/5 complete.
Base fold 2/5 complete.
Base fold 3/5 complete.
Base fold 4/5 complete.
Base fold 5/5 complete.


model,cat_2026,cat_2127,lgb,logit,xgb
fold,,,,,
1,0.972466,0.976220,0.958073,0.898623,0.946809
2,0.966999,0.963263,0.945205,0.890411,0.935243
3,0.963009,0.963636,0.957367,0.898433,0.946708
4,0.969413,0.968789,0.941323,0.894507,0.933208
5,0.974455,0.975078,0.955140,0.893458,0.947664


cat_2127 {'nr': 0.9694055944055944, 'tp': 7763, 'k': 8008, 'p': 80203, 'precision': 0.9694055944055944}
cat_2026 {'nr': 0.9695304695304695, 'tp': 7764, 'k': 8008, 'p': 80203, 'precision': 0.9695304695304695}
xgb {'nr': 0.9414335664335665, 'tp': 7539, 'k': 8008, 'p': 80203, 'precision': 0.9414335664335665}
lgb {'nr': 0.9507992007992008, 'tp': 7614, 'k': 8008, 'p': 80203, 'precision': 0.9507992007992008}
logit {'nr': 0.89498001998002, 'tp': 7167, 'k': 8008, 'p': 80203, 'precision': 0.89498001998002}


## 8. Meta feature matrix

The stacker sees model probabilities, logits, ranks, disagreement summaries, and selected empirical-risk summaries. The full empirical-risk matrix is reserved for the specialist refiner so the global layer does not become a disguised local-rate model.

In [8]:
def safe_logit(p):
    p = np.clip(np.asarray(p, dtype=float), 1e-6, 1-1e-6)
    return np.log(p / (1-p))

meta_oof = pd.DataFrame(index=X.index)
meta_test = pd.DataFrame(index=X_test.index)

for m in BASE_MODELS:
    meta_oof[f'{m}__p'] = base_oof[m]
    meta_test[f'{m}__p'] = base_test[m]
    meta_oof[f'{m}__logit'] = safe_logit(base_oof[m])
    meta_test[f'{m}__logit'] = safe_logit(base_test[m])
    meta_oof[f'{m}__rank'] = pct_rank(base_oof[m])
    meta_test[f'{m}__rank'] = pct_rank(base_test[m])

p_cols = [f'{m}__p' for m in BASE_MODELS]
meta_oof['model_mean'] = meta_oof[p_cols].mean(axis=1)
meta_test['model_mean'] = meta_test[p_cols].mean(axis=1)
meta_oof['model_std'] = meta_oof[p_cols].std(axis=1)
meta_test['model_std'] = meta_test[p_cols].std(axis=1)
meta_oof['model_range'] = meta_oof[p_cols].max(axis=1) - meta_oof[p_cols].min(axis=1)
meta_test['model_range'] = meta_test[p_cols].max(axis=1) - meta_test[p_cols].min(axis=1)

catmean_oof = 0.5*(base_oof['cat_2127'] + base_oof['cat_2026'])
catmean_test = 0.5*(base_test['cat_2127'] + base_test['cat_2026'])
meta_oof['cat_seed_disagreement'] = np.abs(base_oof['cat_2127'] - base_oof['cat_2026'])
meta_test['cat_seed_disagreement'] = np.abs(base_test['cat_2127'] - base_test['cat_2026'])
for other in ['xgb', 'lgb', 'logit']:
    meta_oof[f'catmean_minus_{other}'] = catmean_oof - base_oof[other]
    meta_test[f'catmean_minus_{other}'] = catmean_test - base_test[other]

# Selected EB summaries for the global meta layer.
meta_eb_cols = [
    c for c in EB_COLUMNS
    if any(tag in c for tag in ['risk_a3', 'risk_a10', 'risk_a30', 'log_support', 'purity'])
]
for c in meta_eb_cols:
    meta_oof[c] = eb_oof[c].to_numpy()
    meta_test[c] = eb_test[c].to_numpy()

assert np.isfinite(meta_oof.to_numpy()).all() and np.isfinite(meta_test.to_numpy()).all()
print('Meta features:', meta_oof.shape[1], '| full EB features reserved for refiner:', len(EB_COLUMNS))

Meta features: 57 | full EB features reserved for refiner: 63


## 9. Repeated nested meta/refiner validation

For each of two meta seeds:

1. an outer grouped 5-fold split defines held-out evaluation;
2. inside each outer training partition, a cheap nested 4-fold logistic stack creates out-of-sample stack scores for **refiner training candidate selection**;
3. a stacker fitted on the full outer-training partition scores outer validation and test;
4. candidate claims are selected independently by score percentile in train/validation/test;
5. a supervised CatBoost refiner learns only from high-risk outer-training candidates;
6. its ranking reorders the same candidate score positions, so it cannot pull tail claims into the candidate region;
7. every outer refiner produces its own test prediction. No one-shot full-data refiner is used.

This directly repairs V3's deployment mismatch while retaining the mechanism that produced its boundary gain.

In [9]:
def fit_stack_model(Xm, ym):
    model = Pipeline([
        ('scale', StandardScaler()),
        ('lr', LogisticRegression(C=0.20, solver='lbfgs', max_iter=600, random_state=SEED))
    ])
    model.fit(Xm, ym)
    return model


def select_top_fraction(scores, tie_key, fraction):
    k = int(math.floor(fraction * len(scores)))
    mask = np.zeros(len(scores), dtype=bool)
    if k <= 0:
        return mask
    order = np.lexsort((tie_key, -np.asarray(scores, dtype=float)))
    mask[order[:k]] = True
    return mask


def reorder_candidate_positions(global_scores, ref_scores, candidate_mask, tie_key, weight):
    global_scores = np.asarray(global_scores, dtype=float)
    out = global_scores.copy()
    idx = np.flatnonzero(candidate_mask)
    if len(idx) <= 1:
        return out
    ref_scores = np.asarray(ref_scores, dtype=float)
    if len(ref_scores) != len(idx):
        raise ValueError('Refiner score length does not match candidate count.')
    g_rank = pct_rank(global_scores[idx])
    r_rank = pct_rank(ref_scores)
    blend = (1.0 - weight) * g_rank + weight * r_rank
    # Preserve the exact multiset of global candidate scores; only reorder who receives each position.
    sorted_positions = np.sort(global_scores[idx])
    candidate_order_low_to_high = np.lexsort((tie_key[idx], blend))
    out[idx[candidate_order_low_to_high]] = sorted_positions
    return out


def make_refiner_frame(base_df, row_idx, meta_df, eb_df, stack_scores):
    row_idx = np.asarray(row_idx)
    out = base_df.iloc[row_idx].copy()
    # Base-model evidence and disagreement. Full EB matrix is added below.
    ref_meta_cols = [c for c in meta_df.columns if not c.startswith('eb__')]
    for c in ref_meta_cols:
        out[f'meta__{c}'] = meta_df.iloc[row_idx][c].to_numpy(dtype=np.float32)
    for c in EB_COLUMNS:
        out[c] = eb_df.iloc[row_idx][c].to_numpy(dtype=np.float32)
    out['global_stack_score'] = np.asarray(stack_scores, dtype=np.float32)
    return out


def make_refiner_test_frame(base_df, row_idx, meta_df, eb_df, stack_scores):
    return make_refiner_frame(base_df, row_idx, meta_df, eb_df, stack_scores)

config_keys = [(r, w) for r in CANDIDATE_RATES for w in REFINER_WEIGHTS]
seed_oof_scores = {}
seed_stack_oof = {}
config_test_components = {k: [] for k in config_keys}
stack_test_components = []
fold_detail_rows = []

refiner_params = dict(
    iterations=1200,
    learning_rate=0.035,
    depth=7,
    l2_leaf_reg=10.0,
    random_strength=0.45,
    bagging_temperature=0.6,
    loss_function='Logloss',
    task_type='GPU',
    devices='0:1',
    verbose=False,
    allow_writing_files=False,
)

for meta_seed in META_SEEDS:
    outer = StratifiedGroupKFold(n_splits=META_FOLDS, shuffle=True, random_state=meta_seed)
    outer_splits = list(outer.split(meta_oof, y, groups=signature))
    stack_oof_seed = np.zeros(len(X), dtype=np.float64)
    final_oof_by_cfg = {k: np.zeros(len(X), dtype=np.float64) for k in config_keys}

    for fold, (tr_idx, va_idx) in enumerate(outer_splits, 1):
        # Nested cross-fit stack scores for outer-training rows, used by candidate selection/refiner features.
        tr_stack_oof = np.zeros(len(tr_idx), dtype=np.float64)
        inner = StratifiedGroupKFold(
            n_splits=INNER_META_FOLDS, shuffle=True,
            random_state=meta_seed + fold*101 + 17
        )
        inner_groups = signature[tr_idx]
        for in_tr_rel, in_va_rel in inner.split(meta_oof.iloc[tr_idx], y[tr_idx], groups=inner_groups):
            mdl_inner = fit_stack_model(meta_oof.iloc[tr_idx[in_tr_rel]], y[tr_idx[in_tr_rel]])
            tr_stack_oof[in_va_rel] = mdl_inner.predict_proba(meta_oof.iloc[tr_idx[in_va_rel]])[:, 1]

        # Outer stack model: validation and one complete test component.
        stack_model = fit_stack_model(meta_oof.iloc[tr_idx], y[tr_idx])
        va_stack = stack_model.predict_proba(meta_oof.iloc[va_idx])[:, 1]
        te_stack = stack_model.predict_proba(meta_test)[:, 1]
        stack_oof_seed[va_idx] = va_stack
        stack_test_components.append(te_stack)

        stack_met = nr_at_capacity(y[va_idx], va_stack, TRAIN_TIE[va_idx])

        # One supervised refiner per candidate rate; weights only reorder its outputs afterward.
        for rate in CANDIDATE_RATES:
            tr_mask_rel = select_top_fraction(tr_stack_oof, TRAIN_TIE[tr_idx], rate)
            va_mask_rel = select_top_fraction(va_stack, TRAIN_TIE[va_idx], rate)
            te_mask = select_top_fraction(te_stack, TEST_TIE, rate)

            tr_cand_idx = tr_idx[np.flatnonzero(tr_mask_rel)]
            va_cand_idx = va_idx[np.flatnonzero(va_mask_rel)]
            te_cand_idx = np.flatnonzero(te_mask)

            if len(np.unique(y[tr_cand_idx])) < 2:
                raise RuntimeError('Refiner candidate training set collapsed to one class.')

            ref_tr = make_refiner_frame(
                X, tr_cand_idx, meta_oof, eb_oof,
                tr_stack_oof[np.flatnonzero(tr_mask_rel)]
            )
            ref_va = make_refiner_frame(
                X, va_cand_idx, meta_oof, eb_oof,
                va_stack[np.flatnonzero(va_mask_rel)]
            )
            ref_te = make_refiner_test_frame(
                X_test, te_cand_idx, meta_test, eb_test,
                te_stack[np.flatnonzero(te_mask)]
            )
            ref_cat_cols = [c for c in cat_cols if c in ref_tr.columns]

            refiner = CatBoostClassifier(
                random_seed=meta_seed + fold*1000 + int(rate*100),
                **refiner_params
            )
            perm = np.arange(len(ref_tr))
            np.random.default_rng(meta_seed + fold*10000 + int(rate*1000)).shuffle(perm)
            refiner.fit(ref_tr.iloc[perm], y[tr_cand_idx][perm], cat_features=ref_cat_cols)
            ref_va_p = refiner.predict_proba(ref_va)[:, 1]
            ref_te_p = refiner.predict_proba(ref_te)[:, 1]

            for weight in REFINER_WEIGHTS:
                key = (rate, weight)
                va_final = reorder_candidate_positions(va_stack, ref_va_p, va_mask_rel, TRAIN_TIE[va_idx], weight)
                te_final = reorder_candidate_positions(te_stack, ref_te_p, te_mask, TEST_TIE, weight)
                final_oof_by_cfg[key][va_idx] = va_final
                config_test_components[key].append(te_final)

                final_met = nr_at_capacity(y[va_idx], va_final, TRAIN_TIE[va_idx])
                fold_detail_rows.append({
                    'meta_seed': meta_seed, 'fold': fold, 'candidate_rate': rate,
                    'weight': weight, 'stack_tp': stack_met['tp'], 'final_tp': final_met['tp'],
                    'delta_tp': final_met['tp'] - stack_met['tp'], 'stack_nr': stack_met['nr'],
                    'final_nr': final_met['nr'], 'train_candidates': len(tr_cand_idx),
                    'valid_candidates': len(va_cand_idx), 'test_candidates': len(te_cand_idx)
                })

            del refiner, ref_tr, ref_va, ref_te
            gc.collect()

        print(f'Meta seed {meta_seed} fold {fold}/{META_FOLDS} complete.')

    seed_stack_oof[meta_seed] = stack_oof_seed
    for key in config_keys:
        seed_oof_scores[(meta_seed, key)] = final_oof_by_cfg[key]

fold_detail = pd.DataFrame(fold_detail_rows)
fold_detail.to_csv(OUT_DIR / 'meta_refiner_fold_detail.csv', index=False)
print('Repeated meta/refiner stage complete.')

Meta seed 2026 fold 1/5 complete.
Meta seed 2026 fold 2/5 complete.
Meta seed 2026 fold 3/5 complete.
Meta seed 2026 fold 4/5 complete.
Meta seed 2026 fold 5/5 complete.
Meta seed 9117 fold 1/5 complete.
Meta seed 9117 fold 2/5 complete.
Meta seed 9117 fold 3/5 complete.
Meta seed 9117 fold 4/5 complete.
Meta seed 9117 fold 5/5 complete.
Repeated meta/refiner stage complete.


## 10. Stability-first V5 selection

A candidate must improve both meta seeds to be considered robust. The notebook reports fold wins, worst fold behavior, and a high OOF gate (`0.9815`) as a **submission recommendation**, not as a claim that leaderboard performance is guaranteed.

If no refiner beats the repeated stacker evidence, V5 falls back to stack-only rather than forcing complexity.

In [10]:
selection_rows = []

# Stack-only repeated evidence.
stack_seed_metrics = {}
for s in META_SEEDS:
    met = nr_at_capacity(y, seed_stack_oof[s], TRAIN_TIE)
    stack_seed_metrics[s] = met
selection_rows.append({
    'candidate': 'stack_only', 'candidate_rate': np.nan, 'weight': np.nan,
    'mean_nr': np.mean([stack_seed_metrics[s]['nr'] for s in META_SEEDS]),
    'min_seed_nr': np.min([stack_seed_metrics[s]['nr'] for s in META_SEEDS]),
    'mean_tp': np.mean([stack_seed_metrics[s]['tp'] for s in META_SEEDS]),
    'min_seed_delta_tp_vs_stack': 0.0, 'mean_delta_tp_vs_stack': 0.0,
    'fold_wins': 0, 'worst_fold_delta_tp': 0
})

for key in config_keys:
    rate, weight = key
    seed_metrics = {s: nr_at_capacity(y, seed_oof_scores[(s, key)], TRAIN_TIE) for s in META_SEEDS}
    deltas = [seed_metrics[s]['tp'] - stack_seed_metrics[s]['tp'] for s in META_SEEDS]
    d = fold_detail[(fold_detail['candidate_rate'] == rate) & (fold_detail['weight'] == weight)]
    selection_rows.append({
        'candidate': f'refiner_r{rate:.2f}_w{weight:.2f}',
        'candidate_rate': rate, 'weight': weight,
        'mean_nr': float(np.mean([seed_metrics[s]['nr'] for s in META_SEEDS])),
        'min_seed_nr': float(np.min([seed_metrics[s]['nr'] for s in META_SEEDS])),
        'mean_tp': float(np.mean([seed_metrics[s]['tp'] for s in META_SEEDS])),
        'min_seed_delta_tp_vs_stack': float(np.min(deltas)),
        'mean_delta_tp_vs_stack': float(np.mean(deltas)),
        'fold_wins': int((d['delta_tp'] > 0).sum()),
        'worst_fold_delta_tp': int(d['delta_tp'].min()),
    })

selection = pd.DataFrame(selection_rows)
selection = selection.sort_values(
    ['mean_nr', 'min_seed_nr', 'fold_wins', 'mean_delta_tp_vs_stack'],
    ascending=[False, False, False, False]
).reset_index(drop=True)
selection.to_csv(OUT_DIR / 'meta_selection.csv', index=False)
display(selection)

best = selection.iloc[0].to_dict()
if best['candidate'] == 'stack_only':
    selected_kind = 'stack_only'
    selected_key = None
else:
    selected_kind = 'refiner'
    selected_key = (float(best['candidate_rate']), float(best['weight']))

submission_recommended = bool(
    best['mean_nr'] >= SUBMISSION_NR_GATE
    and (selected_kind == 'stack_only' or best['fold_wins'] >= SUBMISSION_WIN_GATE)
    and best['min_seed_delta_tp_vs_stack'] >= 0
)

print('SELECTED:', best['candidate'])
print('Mean repeated OOF NR@5%:', best['mean_nr'])
print('Submission-worthy gate passed:', submission_recommended)
if not submission_recommended:
    print('Diagnostic note: pipeline still writes a valid V5 submission for inspection, but the predeclared evidence gate did not pass.')

,candidate,candidate_rate,weight,mean_nr,min_seed_nr,mean_tp,min_seed_delta_tp_vs_stack,mean_delta_tp_vs_stack,fold_wins,worst_fold_delta_tp
0,refiner_r0.10_w1.00,0.10,1.00,0.980957,0.980769,7855.5,71.0,72.5,10,7
1,refiner_r0.12_w0.85,0.12,0.85,0.980707,0.980519,7853.5,69.0,70.5,10,4
2,refiner_r0.12_w1.00,0.12,1.00,0.980395,0.980145,7851.0,66.0,68.0,10,8
3,refiner_r0.10_w0.85,0.10,0.85,0.980082,0.979895,7848.5,64.0,65.5,9,0
4,refiner_r0.10_w0.65,0.10,0.65,0.979708,0.979645,7845.5,62.0,62.5,10,6
5,refiner_r0.12_w0.65,0.12,0.65,0.979708,0.979396,7845.5,60.0,62.5,10,4
6,refiner_r0.08_w1.00,0.08,1.00,0.978709,0.978147,7837.5,50.0,54.5,9,-3
7,refiner_r0.08_w0.85,0.08,0.85,0.978584,0.978272,7836.5,51.0,53.5,9,0
8,refiner_r0.08_w0.65,0.08,0.65,0.978084,0.978022,7832.5,49.0,49.5,10,5
9,stack_only,NaN,NaN,0.971903,0.971903,7783.0,0.0,0.0,0,0


SELECTED: refiner_r0.10_w1.00
Mean repeated OOF NR@5%: 0.9809565434565435
Submission-worthy gate passed: False
Diagnostic note: pipeline still writes a valid V5 submission for inspection, but the predeclared evidence gate did not pass.


## 11. Fold-bagged test aggregation and monotone probability mapping

Every selected test score comes from the complete held-out meta/refiner recipes. The ten test components (5 folds × 2 meta seeds) are converted to percentile ranks and averaged, reducing fold-scale sensitivity.

A final one-dimensional logistic map is fitted on combined OOF ranks only to provide continuous `fraud_probability` values. Because the map is monotone when its coefficient is positive, it does not alter the selected ranking that determines the competition metric.

In [11]:
def rank_average(vectors):
    ranks = np.vstack([pct_rank(v) for v in vectors])
    return ranks.mean(axis=0)

if selected_kind == 'stack_only':
    # stack_test_components contains 10 components: 5 folds x 2 seeds.
    final_test_rank = rank_average(stack_test_components)
    final_oof_rank = np.mean(np.vstack([pct_rank(seed_stack_oof[s]) for s in META_SEEDS]), axis=0)
else:
    final_test_rank = rank_average(config_test_components[selected_key])
    final_oof_rank = np.mean(
        np.vstack([pct_rank(seed_oof_scores[(s, selected_key)]) for s in META_SEEDS]), axis=0
    )

final_oof_metric = nr_at_capacity(y, final_oof_rank, TRAIN_TIE)
print('Combined V5 OOF metric:', final_oof_metric)
print('OOF AP:', average_precision_score(y, final_oof_rank))

# Monotone Platt-style mapping on genuinely OOF ranking evidence.
cal = LogisticRegression(C=1000.0, solver='lbfgs', max_iter=1000, random_state=SEED)
cal.fit(final_oof_rank.reshape(-1, 1), y)
coef = float(cal.coef_[0, 0])
if coef > 0:
    final_oof_prob = cal.predict_proba(final_oof_rank.reshape(-1, 1))[:, 1]
    final_test_prob = cal.predict_proba(final_test_rank.reshape(-1, 1))[:, 1]
    calibration_method = 'monotone_logistic'
else:
    # Defensive fallback should the fitted direction be pathological.
    final_oof_prob = np.clip(final_oof_rank, 1e-6, 1-1e-6)
    final_test_prob = np.clip(final_test_rank, 1e-6, 1-1e-6)
    calibration_method = 'rank_identity_fallback'

print('Calibration method:', calibration_method, '| coefficient:', coef)
print('OOF Brier (report-only):', brier_score_loss(y, final_oof_prob))
assert np.isfinite(final_test_prob).all()
assert ((final_test_prob >= 0) & (final_test_prob <= 1)).all()

Combined V5 OOF metric: {'nr': 0.981018981018981, 'tp': 7856, 'k': 8008, 'p': 80203, 'precision': 0.981018981018981}
OOF AP: 0.8273296345016969
Calibration method: monotone_logistic | coefficient: 4.9671941228493734
OOF Brier (report-only): 0.17189243505679977


## 12. Audit-budget diagnostics for the analytical report

The leaderboard uses only 5%, but 3%, 5%, and 7% are retained as report evidence. Fairness/calibration are diagnostics only and do not drive Kaggle model selection under the updated rules.

In [12]:
def age_group(age):
    age = np.asarray(age)
    return np.select(
        [age <= 17, age <= 39, age <= 59],
        ['0-17', '18-39', '40-59'], default='60+'
    )

budget_rows = []
for cap in [0.03, 0.05, 0.07]:
    met = nr_at_capacity(y, final_oof_rank, TRAIN_TIE, cap)
    audit_idx = topk_indices(final_oof_rank, TRAIN_TIE, cap)
    budget_rows.append({
        'capacity': cap, **met,
        'legitimate_audits': int((y[audit_idx] == 0).sum())
    })
audit_table = pd.DataFrame(budget_rows)
audit_table.to_csv(OUT_DIR / 'audit_3_5_7.csv', index=False)
display(audit_table)

# Official protection-style diagnostics at 5% for report use only.
audit_mask = np.zeros(len(y), dtype=bool)
audit_mask[topk_indices(final_oof_rank, TRAIN_TIE, 0.05)] = True
prot_rows = []
for dimension, groups in [
    ('gender', train_raw['jkpst'].astype(str).to_numpy()),
    ('age', age_group(train_raw['umur'].to_numpy()))
]:
    for g in sorted(pd.unique(groups)):
        legit = (y == 0) & (groups == g)
        n_legit = int(legit.sum())
        if n_legit < 100:
            continue
        fp = int((audit_mask & legit).sum())
        prot_rows.append({'dimension': dimension, 'group': g, 'legitimate_n': n_legit, 'audited_legitimate': fp, 'fpr': fp/n_legit})
protection_detail = pd.DataFrame(prot_rows)
protection_detail.to_csv(OUT_DIR / 'protection_detail.csv', index=False)
display(protection_detail)

,capacity,nr,tp,k,p,precision,legitimate_audits
0,0.03,0.986889,4742,4805,80203,0.986889,63
1,0.05,0.981019,7856,8008,80203,0.981019,152
2,0.07,0.971994,10898,11212,80203,0.971994,314


,dimension,group,legitimate_n,audited_legitimate,fpr
0,gender,L,36634,61,0.001665
1,gender,P,43337,91,0.002100
2,age,0-17,20360,30,0.001473
3,age,18-39,20786,48,0.002309
4,age,40-59,23823,39,0.001637
5,age,60+,15002,35,0.002333


## 13. Write and validate the official submission

Only the two required columns are written. `claim_id` stays in the official sample order and is not used as a predictive feature.

In [13]:
submission = sample[['claim_id']].copy()
submission['fraud_probability'] = final_test_prob.astype(np.float64)

assert list(submission.columns) == ['claim_id', 'fraud_probability']
assert len(submission) == len(sample) == len(test_raw)
assert submission['claim_id'].is_unique
assert submission['claim_id'].reset_index(drop=True).equals(sample['claim_id'].reset_index(drop=True))
assert np.isfinite(submission['fraud_probability']).all()
assert submission['fraud_probability'].between(0, 1).all()

SUB_PATH = Path('/kaggle/working/submission_v5.csv')
submission.to_csv(SUB_PATH, index=False)

# Also persist evidence, not alternative leaderboard hacks.
np.savez_compressed(
    OUT_DIR / 'v5_oof_test_scores.npz',
    final_oof_rank=final_oof_rank.astype(np.float32),
    final_oof_probability=final_oof_prob.astype(np.float32),
    final_test_rank=final_test_rank.astype(np.float32),
    final_test_probability=final_test_prob.astype(np.float32),
    y=y.astype(np.int8),
)

selected_record = {
    'selected_candidate': best['candidate'],
    'selected_kind': selected_kind,
    'candidate_rate': None if selected_key is None else selected_key[0],
    'refiner_weight': None if selected_key is None else selected_key[1],
    'mean_repeated_oof_nr5': float(best['mean_nr']),
    'combined_oof_nr5': float(final_oof_metric['nr']),
    'combined_oof_tp_at_5pct': int(final_oof_metric['tp']),
    'submission_recommended_by_predeclared_gate': submission_recommended,
    'calibration_method': calibration_method,
    'official_data_only': True,
    'claim_id_used_as_predictive_feature': False,
    'external_data_or_weights_used': False,
}
with open(OUT_DIR / 'selected_config.json', 'w') as f:
    json.dump(selected_record, f, indent=2)

print('Submission written:', SUB_PATH)
print('Rows:', len(submission))
print('Probability range:', float(submission.fraud_probability.min()), float(submission.fraud_probability.max()))
print(json.dumps(selected_record, indent=2))

Submission written: /kaggle/working/submission_v5.csv
Rows: 40043
Probability range: 0.07731070403794998 0.923119125638004
{
  "selected_candidate": "refiner_r0.10_w1.00",
  "selected_kind": "refiner",
  "candidate_rate": 0.1,
  "refiner_weight": 1.0,
  "mean_repeated_oof_nr5": 0.9809565434565435,
  "combined_oof_nr5": 0.981018981018981,
  "combined_oof_tp_at_5pct": 7856,
  "submission_recommended_by_predeclared_gate": false,
  "calibration_method": "monotone_logistic",
  "official_data_only": true,
  "claim_id_used_as_predictive_feature": false,
  "external_data_or_weights_used": false
}


## 14. Interpretation boundary

This notebook is a reproducible **risk-ranking** pipeline, not an autonomous fraud adjudicator. High scores prioritize human investigation only.

For the report:

- V5's empirical-risk features summarize associations observed only in official training data; they do not establish causal fraud mechanisms.
- Strong geography/facility signal remains a proxy and transport-risk concern.
- The stored training-label order is a documented data-quality artifact and is explicitly neutralized rather than exploited.
- The final Kaggle model is selected only from held-out model evidence. No hidden-label reconstruction or leaderboard row probing is part of the pipeline.